# Aluminium arbitrage — 01 Data pipeline

**Purpose.** Take every input of the netback from its raw file to one point-in-time dataset, a row per route and trade date: the LME, the funding rate and the euro, four premium futures, the duties and surcharges in force, the voyage of each route and the traffic at its straits. Show at each step where a number comes from and when a trader could have read it. The trade statistics that check the netback are loaded here too and read in notebook 03. Notebooks 02 and 03 only read the tables this notebook writes.

**Run order.** `python -m src.download` (optional, appends new dates and new vintages to the files) → this notebook → `02_netback.ipynb` → `03_flows.ipynb`. The two build cells below are the same as `python -m src.sql` and `python -m src.data`.

| Step | Where | Output |
|---|---|---|
| Ingestion | `src/download.py` | `data/lme`, `data/fred`, `data/premiums` (one file per contract), `data/flows` |
| By hand | typed from the documents they cite, or downloaded in a browser | `data/policy`, `data/freight`, `data/press`, `data/portwatch` |
| Raw layer | `src/sql.py` | DuckDB tables `raw_prices`, `raw_flows` |
| Clean layer | `src/sql.py` | DuckDB views `prices`, `futures`, `flows`; the hand files as tables |
| Routes | `src/routes.py` | DuckDB table `routes` |
| Alignment | `src/data.py` | DuckDB table `panel`, read by notebooks 02 and 03 |

In [1]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import pandas as pd

from src import config as c, data, routes, sql

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 80)
plt.rcParams.update({"figure.figsize": (10, 3), "axes.grid": True, "grid.alpha": .3})

## 1. Sources and conventions

Public sources, none behind a key, and files that had to be made by hand because the source refuses a script or is a document.

| Series | Source | Unit | What it is | Readable from |
|---|---|---|---|---|
| `cash`, `three_month` | LME official prices, via Westmetall | USD/t | seller's prices of the second ring, for the cash and three-month prompts | 13:30 London |
| `rate` | effective federal funds rate, New York Fed via FRED (`EFFR`) | % a year | the overnight rate the cargo is financed on, plus a spread | 09:00 New York, the business day after |
| `eurusd` | Fed H.10 noon buying rate via FRED (`DEXUSEU`) | USD per EUR | converts the CBAM price | 12:00 New York, the business day after |
| `AUP` | CME Aluminum MW US Transaction Premium (Platts) futures, via TradingView | USD/lb | Midwest premium, duty-paid, delivered | COMEX settlement, 13:30 New York |
| `EDP`, `AEP` | CME Aluminium European Premium (Fastmarkets) futures, duty-paid and duty-unpaid | USD/t | P1020A in a Rotterdam warehouse, cleared or in bond | same |
| `MJP` | CME Aluminum Japan Premium (Platts) futures | USD/t | spot premium CIF Japanese main ports | same |
| `tariffs` | section 232 proclamations, Federal Register | rate | US duty by origin, with signing and effective dates | the signing date |
| `cbam_price` | European Commission, price of CBAM certificates | EUR/tCO2e | quarterly certificate price for 2026 | its publication date |
| `war_surcharges` | carrier notices, as reported by the trade press | USD per box | Gulf and Red Sea surcharges | the effective date |
| `press` | Fastmarkets and Platts assessments quoted in the press | USD/t | a check on the futures, not an input | the publication date |
| `transits` | IMF PortWatch, daily chokepoint transit calls, downloaded in a browser | ships a day | traffic at Suez, Bab el-Mandeb, Malacca, Panama, Gibraltar, the Cape | Tuesday 09:00 New York, the week to the Sunday before |
| `flows` | UN Comtrade (US, Japan, Canada), Eurostat Comext (EU-27), HS 7601 | t a month | imports and exports of unwrought aluminium | each month's first release |

Each premium future settles on the average, over its contract month, of a price agency's daily assessment. The contract of a month is therefore the market's price for metal delivered in that month, which is how the netback uses it.

Four dates are kept apart:

- **Observation date** — `obs_date`, the day or month the source stamps on the number.
- **Availability** — the `_utc` stamps and publication dates: when the number could be read. These are what the joins use.
- **Ingestion** — `ingested_at`, when the file holding the row was downloaded.
- **Vintage** — prices are final once printed: the first print is kept and a later difference is reported at download. Trade statistics are revised: every release downloaded is kept as its own row (section 9).

## 2. Ingestion: raw files into DuckDB

`sql.build()` loads the price files into `raw_prices` in long form, one row per file, series and date, and the trade files into `raw_flows`, one row per file, partner, month and release. Both primary keys leave an existing row alone, so the load is idempotent: the cell below can be run any number of times.

In [2]:
sql.build()

81,231 rows added to raw_prices, 81,231 in total
10,327 rows added to raw_flows, 10,327 in total
20 rows in tariffs
3 rows in cbam_price
4 rows in war_surcharges
5 rows in press
1 rows in mjp_quarterly


80,528 rows in transits


In [3]:
sql.build()  # a second run adds nothing

0 rows added to raw_prices, 81,231 in total
0 rows added to raw_flows, 10,327 in total
20 rows in tariffs
3 rows in cbam_price
4 rows in war_surcharges
5 rows in press
1 rows in mjp_quarterly


80,528 rows in transits


In [4]:
sql.query("DESCRIBE raw_prices")[["column_name", "column_type", "key"]]

,column_name,column_type,key
0,file,VARCHAR,PRI
1,series,VARCHAR,PRI
2,obs_date,DATE,PRI
3,value,DOUBLE,NaN
4,ingested_at,TIMESTAMP,NaN


In [5]:
# what entered the database, and when, by folder
sql.query("""
    SELECT split_part(file, '/', 1) AS folder, series, COUNT(DISTINCT file) AS files, CAST(MAX(ingested_at) AS DATE) AS ingested,
           COUNT(*) AS observations, MIN(obs_date) AS first_date, MAX(obs_date) AS last_date
    FROM raw_prices GROUP BY 1, 2 ORDER BY 1, 2
""")

,folder,series,files,ingested,observations,first_date,last_date
0,fred,eurusd,1,2026-11-18,6988,1999-01-04,2026-11-13
1,fred,rate,1,2026-11-18,6622,2000-07-03,2026-11-17
2,lme,cash,1,2026-11-18,4773,2008-01-02,2026-11-17
3,lme,stock,1,2026-11-18,4773,2008-01-02,2026-11-17
4,lme,three_month,1,2026-11-18,4692,2008-01-02,2026-11-17
5,premiums,settle,537,2026-11-18,53383,2015-09-07,2026-11-17


## 3. Clean layer

`prices` and `futures` are SQL views on `raw_prices`, so they cannot drift from the raw data. They only standardise: each number gets its unit, a contract file name becomes its product and delivery month, and the one LME row dated on a Saturday is left out. No value is repaired or filled. The hand files are tables, each row with the document it was read from.

In [6]:
sql.query("""
    SELECT series, unit, COUNT(*) AS observations, MIN(obs_date) AS first_date, MAX(obs_date) AS last_date,
           MIN(value) AS lowest, MAX(value) AS highest
    FROM prices GROUP BY 1, 2 ORDER BY 1
""")

,series,unit,observations,first_date,last_date,lowest,highest
0,cash,USD/t,4772,2008-01-02,2026-11-17,"1,253.50","3,984.50"
1,eurusd,USD/EUR,6988,1999-01-04,2026-11-13,0.83,1.60
2,rate,% a year,6622,2000-07-03,2026-11-17,0.04,7.03
3,stock,t,4772,2008-01-02,2026-11-17,"220,350.00","5,492,325.00"
4,three_month,USD/t,4691,2008-01-02,2026-11-17,"1,288.50","3,968.00"


In [7]:
# one file per contract: AUPZ2026.csv is the December 2026 Midwest contract
sql.query("""
    SELECT product, unit, COUNT(DISTINCT contract) AS contracts, MIN(delivery_month) AS first_month, MAX(delivery_month) AS last_month,
           COUNT(*) AS settles, MIN(settle) AS lowest, MAX(settle) AS highest
    FROM futures GROUP BY 1, 2 ORDER BY 1
""")

,product,unit,contracts,first_month,last_month,settles,lowest,highest
0,AEP,USD/t,135,2016-01-01,2027-03-01,13487,60.00,519.29
1,AUP,USD/lb,135,2016-01-01,2027-03-01,13497,0.06,1.20
2,EDP,USD/t,132,2016-04-01,2027-03-01,13027,88.67,630.00
3,MJP,USD/t,135,2016-01-01,2027-03-01,13372,58.19,374.29


In [8]:
# the section 232 changes that matter for primary aluminium, by origin group
sql.query("SELECT origin_group, signed, effective, rate, document, note FROM tariffs ORDER BY origin_group, signed")

,origin_group,signed,effective,rate,document,note
0,canada,2016-01-01,2016-01-01,0.00,HTSUS 7601.10.60,general rate Free: no duty on primary aluminium before section 232
1,canada,2018-05-31,2018-06-01,0.10,Proclamation 9758,Canada's exemption ends on 1 June 2018
2,canada,2019-05-19,2019-05-20,0.00,Proclamation 9893,Canada and Mexico exempted again
3,canada,2020-08-06,2020-08-16,0.10,Proclamation 10060,10% re-imposed on Canadian non-alloyed unwrought aluminium (7601.10)
4,canada,2020-10-27,2020-09-01,0.00,Proclamation 10106,exemption restored for goods entered from 1 September 2020: retroactive
5,canada,2025-02-10,2025-03-12,0.25,Proclamation 10895,25% from all countries; every exemption ends
6,canada,2025-06-03,2025-06-04,0.50,Proclamation 10947,50% from 4 June 2025
7,canada,2026-04-02,2026-04-06,0.50,Proclamation 11021,50% on the full customs value regardless of metal content: unchanged for pri...
8,other,2016-01-01,2016-01-01,0.00,HTSUS 7601.10.60,general rate Free: no duty on primary aluminium before section 232
9,other,2018-03-08,2018-03-23,0.10,Proclamation 9704,10% on aluminium articles from all countries except Canada and Mexico


In [9]:
sql.query("SELECT * EXCLUDE (url, ingested_at) FROM cbam_price")

,quarter,eur_per_tco2,published
0,2026Q1,75.36,2026-04-07
1,2026Q2,75.28,2026-07-06
2,2026Q3,82.32,2026-10-05


In [10]:
sql.query("SELECT effective, area, carrier, usd_per_teu, note FROM war_surcharges")

,effective,area,carrier,usd_per_teu,note
0,2016-01-01,gulf,none,0,no carrier surcharge on file before the 2026 Gulf war
1,2016-01-01,red_sea,none,0,no carrier surcharge on file for a Red Sea transit before October 2026
2,2026-03-02,gulf,Hapag-Lloyd,1500,war risk surcharge on cargo to and from the Upper Gulf / Arabian Gulf / Pers...
3,2026-10-12,red_sea,MSC,91,piracy risk (55) and Suez Canal (36) surcharges extended through October 202...


Each hand file carries its source on every row (the `url` column, left out of the displays). The section 232 file keeps two dates per change, the day the proclamation was signed and the day it applies to entries: section 6 uses both. The zero rows of the surcharge file say that no carrier notice was found before 2026, not that none existed.

## 4. Validation of the raw data

Each check is one query. A check that finds something is followed by the decision taken.

In [11]:
# missing values, duplicate keys, weekend dates, in the raw price table
sql.query("""
    SELECT split_part(file, '/', 1) AS folder, COUNT(*) FILTER (WHERE value IS NULL OR obs_date IS NULL) AS missing,
           COUNT(*) - COUNT(DISTINCT (file, series, obs_date)) AS duplicates,
           COUNT(*) FILTER (WHERE isodow(obs_date) > 5) AS weekend_dates
    FROM raw_prices GROUP BY 1 ORDER BY 1
""")

,folder,missing,duplicates,weekend_dates
0,fred,0,0,0
1,lme,0,0,3
2,premiums,0,0,0


In [12]:
# the LME in the last weeks: the mid-November squeeze
lme = sql.query("""
    SELECT obs_date, cash, three_month, cash - three_month AS backwardation
    FROM (PIVOT (SELECT series, obs_date, value FROM prices WHERE series IN ('cash', 'three_month')) ON series USING MAX(value))
    ORDER BY obs_date
""").set_index("obs_date")
lme.tail(8)

,cash,three_month,backwardation
obs_date,,,
2026-11-06,"3,117.00","3,050.00",67.00
2026-11-09,"3,147.00","3,059.50",87.50
2026-11-10,"3,137.00","3,027.00",110.00
2026-11-11,"3,223.50","3,064.00",159.50
2026-11-12,"3,155.50","3,032.00",123.50
2026-11-13,"3,150.50","3,018.00",132.50
2026-11-16,"3,332.00","3,020.00",312.00
2026-11-17,"3,060.50","2,940.00",120.50


In [13]:
# the widest backwardations on file since the sample starts
lme.loc[c.SAMPLE_START:].nlargest(5, "backwardation")

,cash,three_month,backwardation
obs_date,,,
2026-11-16,"3,332.00","3,020.00",312.00
2026-11-11,"3,223.50","3,064.00",159.50
2026-11-13,"3,150.50","3,018.00",132.50
2026-11-12,"3,155.50","3,032.00",123.50
2026-11-17,"3,060.50","2,940.00",120.50


- **Three LME rows are dated on a Saturday**, in 2008, copies of the Friday before: they stay in `raw_prices` and the `prices` view leaves them out.
- **The squeeze of mid-November is kept and flagged.** On 16 November cash printed 3,332 USD/t, 312 over three months, the widest backwardation since the sample starts, and fell back to 3,060.5 the next day while three months barely moved. Only the nearby leg jumping is what a squeeze of the nearby dates does, and the market was reported squeezed that week; without a second source it cannot be told from a bad print, so notebook 02 reads that day with care.

In [14]:
# stale marks: share of days a contract settles exactly where it settled the day before, by product and year
f = sql.query("SELECT product, contract, trade_date, settle FROM futures ORDER BY contract, trade_date")
f["unchanged"] = f.groupby("contract")["settle"].diff() == 0
(100 * f.groupby([f["trade_date"].dt.year.rename("year"), "product"])["unchanged"].mean()).unstack().loc[2016:].T.round(0)

year,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
product,,,,,,,,,,,
AEP,69.00,80.00,81.00,81.00,83.00,67.00,67.00,67.00,68.00,70.00,73.00
AUP,59.00,59.00,54.00,59.00,49.00,33.00,35.00,43.00,40.00,21.00,20.00
EDP,85.00,89.00,80.00,84.00,77.00,63.00,52.00,49.00,49.00,38.00,39.00
MJP,69.00,72.00,84.00,86.00,88.00,82.00,84.00,77.00,78.00,75.00,80.00


In [15]:
# holes in a contract's life: the longest gaps between two settles, in calendar days
f["gap"] = f.groupby("contract")["trade_date"].diff().dt.days
f.groupby("product")["gap"].agg(["median", "max"])

,median,max
product,,
AEP,1.00,5.00
AUP,1.00,5.00
EDP,1.00,5.00
MJP,1.00,20.00


- **The premium futures are marks.** The duty-unpaid (AEP) and Japan (MJP) contracts settle unchanged on 70 to 80% of days in most years; the Midwest (AUP) and duty-paid (EDP) contracts have become livelier since 2021, unchanged on 20 to 40% of days in 2025-26. An unchanged settle is a mark carried by the exchange, not a missing value: it is kept, and notebook 02 reads small moves with that in mind.
- **No contract has a hole longer than a long weekend**, except the January 2025 Japan contract, 20 days without a settle in September 2024, months before its arrival window.

In [16]:
# a check on the futures: each press assessment against the contract of its month, settled on that day or the last before
press = sql.query("SELECT published, market, low, high, agency FROM press ORDER BY published")
press["mid"] = (press["low"] + press["high"]) / 2
product = {"rotterdam_dp": c.PREMIUMS["rotterdam"], "rotterdam_du": c.DUTY_UNPAID, "japan": c.PREMIUMS["japan"]}
press["future"] = [f[(f["product"] == product[m]) & (f["contract"].str[-5:] == f"{c.MONTH_CODES[d.month - 1]}{d.year}") & (f["trade_date"] <= d)]["settle"].iloc[-1]
                   for m, d in zip(press["market"], press["published"])]
press["future_minus_mid"] = press["future"] - press["mid"]
press

,published,market,low,high,agency,mid,future,future_minus_mid
0,2025-12-18,rotterdam_dp,330,330,Platts,330.00,325.96,-4.04
1,2025-12-18,rotterdam_du,280,280,Platts,280.00,277.90,-2.10
2,2026-10-13,rotterdam_dp,550,585,Fastmarkets,567.50,559.67,-7.83
3,2026-10-13,japan,237,237,Platts,237.00,240.63,3.63
4,2026-10-27,rotterdam_dp,560,590,Fastmarkets,575.00,567.02,-7.98


The futures follow the assessments they settle on: the duty-paid contract of the quote's month is 4 to 8 USD/t under the press mid, the duty-unpaid 2 under, Japan 4 over, which is what an average over the month in progress against one day's assessment should give. Five quotes are a check, not a validation: the agencies' daily series are paid, and the press quotes them rarely.

In [17]:
# unit check on the Midwest: the future in USD/lb against the same premium in USD/t
aup = f[f["product"] == "AUP"].groupby("trade_date")["settle"].last()
pd.DataFrame({"USD/lb": aup.loc["2026-11-10":], "USD/t": aup.loc["2026-11-10":] * c.LB_PER_T})

,USD/lb,USD/t
trade_date,,
2026-11-10,1.03,"2,281.78"
2026-11-11,1.02,"2,255.79"
2026-11-12,1.02,"2,254.23"
2026-11-13,1.02,"2,254.23"
2026-11-16,1.03,"2,270.25"
2026-11-17,1.03,"2,276.27"


The Midwest is quoted in US dollars per pound: 1.02 USD/lb is 2,254 USD/t, the figure of the brief. Every other premium is in USD per tonne.

In [18]:
# PortWatch: what the file holds, for the passages the routes use
transits = sql.query(f"SELECT portname, COUNT(*) AS days, MIN(date) AS first, MAX(date) AS last, AVG({c.TRANSITS}) AS mean_container_ships, "
                     f"COUNT(*) FILTER (WHERE {c.TRANSITS} IS NULL) AS missing FROM transits WHERE portname IN {tuple(c.PASSAGES.values()) + ('Strait of Hormuz',)} GROUP BY 1 ORDER BY 1")
transits

,portname,days,first,last,mean_container_ships,missing
0,Bab el-Mandeb Strait,2876,2019-01-01,2026-11-15,12.99,0
1,Cape of Good Hope,2876,2019-01-01,2026-11-15,10.66,0
2,Gibraltar Strait,2876,2019-01-01,2026-11-15,31.94,0
3,Malacca Strait,2876,2019-01-01,2026-11-15,50.94,0
4,Panama Canal,2876,2019-01-01,2026-11-15,7.51,0
5,Strait of Hormuz,2876,2019-01-01,2026-11-15,12.61,0
6,Suez Canal,2876,2019-01-01,2026-11-15,14.52,0


In [19]:
# container-ship transits by quarter against their 2019 average
t = sql.query(f"SELECT date, portname, {c.TRANSITS} AS ships FROM transits WHERE portname IN ('Bab el-Mandeb Strait', 'Suez Canal', 'Cape of Good Hope', 'Strait of Hormuz')")
t["date"] = pd.to_datetime(t["date"])
t = t.pivot(index="date", columns="portname", values="ships")
(t.resample("QE").mean() / t.loc["2019"].mean()).loc["2023":].round(2)

portname,Bab el-Mandeb Strait,Cape of Good Hope,Strait of Hormuz,Suez Canal
date,,,,
2023-03-31,1.05,0.97,1.37,1.02
2023-06-30,1.16,1.02,1.72,1.17
2023-09-30,1.23,1.10,1.72,1.22
2023-12-31,0.96,1.25,1.29,1.06
2024-03-31,0.26,3.12,1.29,0.51
2024-06-30,0.30,3.37,1.71,0.51
2024-09-30,0.33,3.48,1.77,0.50
2024-12-31,0.39,3.44,1.55,0.52
2025-03-31,0.44,3.48,1.47,0.53


- **The PortWatch file is complete**: the seven passages the routes use, every day from 1 January 2019 to 15 November 2026, no value missing.
- **Container ships left the Red Sea in December 2023 and have not come back.** Transits at Bab el-Mandeb fell from 0.96 of their 2019 level in the last quarter of 2023 to 0.26 in the first of 2024, and have stayed between 0.26 and 0.44 since; the Cape took three times its 2019 traffic. Hormuz fell to 0.08 of 2019 in the second quarter of 2026 and 0.03 in the latest weeks. Container ships only: tankers and bulkers fell much less, which is why the route rule counts container ships (notebook 02).

## 5. Point-in-time stamps

A date alone does not say when a number was readable. The LME prints at lunchtime in London, COMEX settles in the New York afternoon, the funding rate and the euro of a day only come out the next morning in New York, and PortWatch publishes a whole week at once. Each number is stamped with its source's publication time converted to UTC, and each row is anchored at the New York close.

In [20]:
pd.DataFrame(c.CLOSE, index=["time zone", "local time"]).T

,time zone,local time
cash,Europe/London,13:30:00
three_month,Europe/London,13:30:00
rate,America/New_York,09:00:00
eurusd,America/New_York,12:00:00
AUP,America/New_York,13:30:00
EDP,America/New_York,13:30:00
AEP,America/New_York,13:30:00
MJP,America/New_York,13:30:00


In [21]:
# the numbers dated on one summer and one winter day, on one clock. The rate and the euro dated D are stamped on D+1
day = {name: data.series(name).set_index(f"{name}_date")[f"{name}_utc"] for name in ("cash", "rate", "eurusd")}
day["AUP"] = data.futures("AUP").groupby("trade_date")["settle_utc"].first()
pd.DataFrame(day).loc[["2026-06-10", "2026-01-15"]].T

,2026-06-10,2026-01-15
cash,2026-06-10 12:30:00+00:00,2026-01-15 13:30:00+00:00
rate,2026-06-11 13:00:00+00:00,2026-01-16 14:00:00+00:00
eurusd,2026-06-11 16:00:00+00:00,2026-01-16 17:00:00+00:00
AUP,2026-06-10 17:30:00+00:00,2026-01-15 18:30:00+00:00


On 10 June 2026, London on summer time, the LME prints at 12:30 UTC and COMEX settles at 17:30 UTC; the funding rate and the euro of that day only come out the next day. Whatever the season every number dated D is out before the New York close of D, except the rate and the euro, which a row can only carry from an earlier day.

In [22]:
c.ASOF  # the anchor of a row: the New York close on its trade date

('America/New_York', '17:00:00')

In [23]:
# PortWatch's calendar against the download: the file was saved on the Wednesday after a release. Under the rule, the week
# to Sunday 15 November is out (released Tuesday 17 November) and Monday 16 November is not
share = data.traffic()
downloaded = sql.query("SELECT MAX(ingested_at) AS t FROM transits")["t"].iloc[0].tz_localize("UTC")
newest = share["date"].max()
release = share.loc[share["date"] == newest, "published_utc"].iloc[0]
assert release <= downloaded and newest + pd.Timedelta(days=1) + pd.Timedelta(days=8) > downloaded.tz_localize(None)
print(f"downloaded {downloaded:%a %d %b %Y %H:%M} UTC, newest day {newest:%a %d %b}, released {release:%a %d %b %H:%M} UTC")

downloaded Wed 18 Nov 2026 18:14 UTC, newest day Sun 15 Nov, released Tue 17 Nov 14:00 UTC


The download agrees with the calendar: saved on Wednesday 18 November, the file ends on Sunday 15 November, the week released on Tuesday 17 November. One download tests one release; the times are PortWatch's published schedule, not stamps recorded with each count.

## 6. Alignment

**Which ports and how far.** `routes.build()` sends each origin to each destination on searoute's maritime network. Each port is given as a coordinate; searoute snaps it to the nearest port of its database. The port it chose is printed with its UN/LOCODE and checked against the one intended. The voyages are stored as the table `routes`, which the alignment reads.

In [24]:
r = routes.build()
sql.write("routes", r)
r[["origin", "destination", "variant", "port_from", "locode_from", "port_to", "locode_to", "distance_nm", "sea_days", "passages", "freight_sea", "land_usd", "land_days"]]

,origin,destination,variant,port_from,locode_from,port_to,locode_to,distance_nm,sea_days,passages,freight_sea,land_usd,land_days
0,quebec,midwest,rail,NaN,NaN,NaN,NaN,0.00,0.00,,0.00,80,7
1,quebec,rotterdam,direct,Sept-Iles,CASEI,Rotterdam,NLRTM,"2,735.08",8.14,,40.45,0,0
2,quebec,japan,direct,Sept-Iles,CASEI,Yokohama,JPYOK,"10,765.84",32.04,panama,159.22,0,0
3,iceland,midwest,direct,Grundartangi,ISGRT,Baltimore,USBAL,"2,822.75",8.40,,41.75,40,10
4,iceland,rotterdam,direct,Grundartangi,ISGRT,Rotterdam,NLRTM,"1,240.39",3.69,,18.34,0,0
5,iceland,japan,direct,Grundartangi,ISGRT,Yokohama,JPYOK,"11,865.93",35.32,"babalmandab,malacca,suez,gibraltar",175.49,0,0
6,iceland,japan,detour,Grundartangi,ISGRT,Yokohama,JPYOK,"12,076.39",35.94,panama,178.60,0,0
7,gulf,midwest,direct,Khor Al Fakkan,AEKLF,Baltimore,USBAL,"8,246.51",24.54,"gibraltar,suez,babalmandab",121.96,40,10
8,gulf,midwest,detour,Khor Al Fakkan,AEKLF,Baltimore,USBAL,"11,658.34",34.70,south_africa,172.42,40,10
9,gulf,rotterdam,direct,Khor Al Fakkan,AEKLF,Rotterdam,NLRTM,"6,139.11",18.27,"gibraltar,suez,babalmandab",90.79,0,0


In [25]:
# the ports intended, by UN/LOCODE
intended = {"quebec": "CASEI", "iceland": "ISGRT", "gulf": "AEKLF", "india": "INPRT", "malaysia": "MYBTU"}
sea = r[r["variant"] != "rail"]
assert (sea["locode_from"] == sea["origin"].map(intended)).all()
assert set(sea["locode_to"]) == {"USBAL", "NLRTM", "JPYOK"}

- **Every port is the one intended**: Sept-Îles (CASEI), Grundartangi (ISGRT), Khor Fakkan (AEKLF), Paradip (INPRT), Bintulu (MYBTU), and Rotterdam (NLRTM), Yokohama (JPYOK) and Baltimore (USBAL). Rotterdam is given at its own coordinate: a point at sea off the Maasvlakte snaps to Hook of Holland instead.
- **Baltimore is nearer than New Orleans from every sea origin**, Malaysia included, through Suez, so the Midwest is reached through Baltimore.
- **Seven voyages cross Bab el-Mandeb** and get a detour: round the Cape for the Gulf and India, through Panama for Malaysia to the Midwest and Iceland to Japan, round the Cape and through the Sunda Strait for Malaysia to Rotterdam. The detour adds 0.6 to 14 days at sea.

**Which premium.** A cargo arranged on trade date D arrives on D plus its days at sea and on land, and is sold on the premium of that month: the contract of the arrival month, settled at or before the anchor.

In [26]:
df = data.align("gulf", "rotterdam", "direct")
df.shape

(2681, 31)

In [27]:
# the join by hand for the duty-paid leg on the last five trade dates
anchors = df[["asof_utc", "arrival", "delivery_month"]].tail(5).reset_index()
edp = data.futures("EDP").sort_values("settle_utc")
by_hand = pd.merge_asof(anchors, edp, left_on="asof_utc", right_on="settle_utc", left_by="delivery_month", right_by="delivery_month",
                        direction="backward", tolerance=pd.Timedelta(c.TOLERANCE))
by_hand

,trade_date_x,asof_utc,arrival,delivery_month,contract,trade_date_y,settle,settle_utc
0,2026-11-11,2026-11-11 22:00:00+00:00,2026-11-30,2026-11-01,EDPX2026,2026-11-11,570.14,2026-11-11 18:30:00+00:00
1,2026-11-12,2026-11-12 22:00:00+00:00,2026-12-01,2026-12-01,EDPZ2026,2026-11-12,491.25,2026-11-12 18:30:00+00:00
2,2026-11-13,2026-11-13 22:00:00+00:00,2026-12-02,2026-12-01,EDPZ2026,2026-11-13,456.67,2026-11-13 18:30:00+00:00
3,2026-11-16,2026-11-16 22:00:00+00:00,2026-12-05,2026-12-01,EDPZ2026,2026-11-16,456.67,2026-11-16 18:30:00+00:00
4,2026-11-17,2026-11-17 22:00:00+00:00,2026-12-06,2026-12-01,EDPZ2026,2026-11-17,439.68,2026-11-17 18:30:00+00:00


In [28]:
# the same numbers data.align carries
assert (by_hand["settle"].to_numpy() == df["premium_settle"].tail(5).to_numpy()).all()
assert (by_hand["contract"].to_numpy() == df["premium_contract"].tail(5).to_numpy()).all()

On 12 November the Gulf's arrival date crossed into December and the premium rolled from the November contract (570 USD/t) to December (491): in the November backwardation of the premium curve, a cargo that lands a few days later sells lower.

**Which duty.** A section 232 rate applies to a cargo if it was proclaimed by the trade date and is in force on the arrival date. Two episodes show why both dates are needed: a proclamation signed in October 2020 restored Canada's exemption retroactively to 1 September, and an exemption granted to the UAE in January 2021 was revoked before it took effect.

In [29]:
quebec = data.align("quebec", "midwest", "rail")
quebec.loc["2020-08-03":"2020-11-03", ["arrival", "us_tariff"]].iloc[::6]

,arrival,us_tariff
trade_date,,
2020-08-03,2020-08-10,0.00
2020-08-11,2020-08-18,0.10
2020-08-19,2020-08-26,0.10
2020-08-27,2020-09-03,0.10
2020-09-07,2020-09-14,0.10
2020-09-15,2020-09-22,0.10
2020-09-23,2020-09-30,0.10
2020-10-01,2020-10-08,0.10
2020-10-09,2020-10-16,0.10


In [30]:
gulf = data.align("gulf", "midwest", "direct")
gulf.loc["2021-01-14":"2021-02-05", ["arrival", "us_tariff"]]

,arrival,us_tariff
trade_date,,
2021-01-14,2021-02-18,0.10
2021-01-15,2021-02-19,0.10
2021-01-18,2021-02-22,0.10
2021-01-19,2021-02-23,0.00
2021-01-20,2021-02-24,0.00
2021-01-21,2021-02-25,0.00
2021-01-22,2021-02-26,0.00
2021-01-25,2021-03-01,0.00
2021-01-26,2021-03-02,0.00


- **Canada, 2020.** From 16 August Canadian metal paid 10%; Proclamation 10106, signed on 27 October, restored the exemption for entries from 1 September. A trader in September did not know it: the netback carries 10% until 26 October and 0% from 27 October, never the retroactive 0% before it was signed.
- **UAE, 2021.** Proclamation 10139, signed on 19 January, exempted the UAE from 3 February; Proclamation 10144 revoked it on 1 February, before it took effect. For nine trade days the exemption was known and in force by the arrival dates, and the netback carries it.

**Which road.** The route rule reads the traffic at Bab el-Mandeb as PortWatch published it by the anchor: the week to the latest Sunday released, against 2019.

In [31]:
df.loc["2023-12-14":"2024-01-04", ["bab_el_mandeb", "traffic"]]

,bab_el_mandeb,traffic
trade_date,,
2023-12-14,0.87,0.87
2023-12-15,0.87,0.87
2023-12-18,0.87,0.87
2023-12-19,0.84,0.84
2023-12-20,0.84,0.84
2023-12-21,0.84,0.84
2023-12-22,0.84,0.84
2023-12-27,0.23,0.23
2023-12-28,0.23,0.23


Traffic changes only on Tuesdays, the release days, and the detour switches on 27 December 2023: the first release after the container lines left the Red Sea showed the week to 24 December at 0.23 of 2019.

## 7. Validation after alignment

A backward join only guarantees that the matched stamp is not later than the anchor. The assertions below are the conditions notebook 02 relies on, on every route.

In [32]:
panel = {key: data.align(*key) for key in sql.query("SELECT origin, destination, variant FROM routes").itertuples(index=False)}
for key, p in panel.items():
    # no look-ahead: every price and every publication on a row was out at or before its anchor
    assert (p["price_utc"] <= p["asof_utc"]).all() and (p["rate_utc"] <= p["asof_utc"]).all()
    assert (p["eurusd_utc"] <= p["asof_utc"]).all() and (p["premium_settle_utc"] <= p["asof_utc"]).all()
    assert (p["premium_trade_date"] <= p.index).all()
    if "premium_du_settle_utc" in p and p["premium_du_settle_utc"].notna().any():
        assert (p["premium_du_settle_utc"] <= p["asof_utc"]).all()
    assert (p["cbam_published"].dropna() <= p.index[p["cbam_published"].notna()]).all()
    # the contract is that of the month the metal arrives
    assert (p["premium_contract"].str[-5:] == [f"{c.MONTH_CODES[m.month - 1]}{m.year}" for m in p["arrival"]]).all()
    # one row per trade date, in order, no price leg older than the tolerance
    assert p.index.is_unique and p.index.is_monotonic_increasing
    assert p["staleness_days"].max() <= pd.Timedelta(c.TOLERANCE).days
len(panel)

22

In [33]:
# PortWatch's traffic on a row is from a release out before the anchor: the newest day it can hold is the Sunday before
# the latest Tuesday release
p = panel[("gulf", "rotterdam", "direct")]
latest_sunday = (p["asof_utc"].dt.tz_convert("America/New_York") - pd.Timedelta(hours=9)).dt.normalize().dt.tz_localize(None)
latest_sunday = latest_sunday - pd.to_timedelta((latest_sunday.dt.dayofweek - 1) % 7 + 2, unit="D")
share = data.traffic()
bab = share[share["portname"] == "Bab el-Mandeb Strait"].set_index("date")["share"]
check = p.loc["2019-02":, "bab_el_mandeb"]
assert (check.to_numpy() == bab.reindex(latest_sunday.loc[check.index]).to_numpy()).all()

In [34]:
# staleness: age in days of the oldest price leg on a row, all routes
pd.concat({key: p["staleness_days"] for key, p in panel.items()}).value_counts().sort_index().rename("rows").to_frame().T

staleness_days,1,2,3,4,5
rows,45175,1276,9642,2932,66


No row has a price leg older than the tolerance. A day of staleness is the floor, since the rate and the euro of a day are only out the next day; three to five days are weekends and US holidays, and the euro, which FRED posts a few days late.

In [35]:
# missing matches: LME trade dates of the sample that did not make it into a route
grid = data.market()["trade_date"]
dropped = {"-".join(key): grid[~grid.isin(p.index)].dt.strftime("%Y-%m-%d").tolist() for key, p in panel.items()}
{key: (len(days), days[:4]) for key, days in dropped.items() if days}

{'quebec-rotterdam-direct': (15,
  ['2016-04-01', '2016-04-04', '2016-04-05', '2016-04-06']),
 'iceland-rotterdam-direct': (18,
  ['2016-04-01', '2016-04-04', '2016-04-05', '2016-04-06']),
 'gulf-rotterdam-direct': (7,
  ['2016-04-01', '2016-04-04', '2016-04-05', '2016-04-06']),
 'india-rotterdam-direct': (4,
  ['2016-04-01', '2016-04-04', '2016-04-05', '2016-04-06']),
 'malaysia-rotterdam-direct': (1, ['2016-04-01'])}

Only Rotterdam's direct routes lose dates, 1 to 18 of them, all in April 2016. The April 2016 duty-paid contract printed once, on 14 March 2016: a voyage short enough to land in April has no settle within the tolerance and is dropped, until its arrival month is May.

## 8. Final dataset

`data.build()` stores the routes and the aligned frames as the tables `routes` and `panel`. Reading a route back gives exactly the frame validated above.

In [36]:
data.build()

,origin,destination,variant,trade_dates,first,last
0,gulf,japan,direct,2688,2016-04-01,2026-11-17
1,gulf,midwest,detour,2688,2016-04-01,2026-11-17
2,gulf,midwest,direct,2688,2016-04-01,2026-11-17
3,gulf,rotterdam,detour,2688,2016-04-01,2026-11-17
4,gulf,rotterdam,direct,2681,2016-04-12,2026-11-17
5,iceland,japan,detour,2688,2016-04-01,2026-11-17
6,iceland,japan,direct,2688,2016-04-01,2026-11-17
7,iceland,midwest,direct,2688,2016-04-01,2026-11-17
8,iceland,rotterdam,direct,2670,2016-04-27,2026-11-17
9,india,japan,direct,2688,2016-04-01,2026-11-17


In [37]:
# the stored table holds every route's columns: read back in the order align made them
for key, p in panel.items():
    assert data.dataset(*key)[p.columns].equals(p), key

In [38]:
data.dataset("gulf", "rotterdam", "direct").tail(2).T

trade_date,2026-11-16,2026-11-17
cash,"3,332.00","3,060.50"
three_month,"3,020.00","2,940.00"
price_utc,2026-11-16 13:30:00+00:00,2026-11-17 13:30:00+00:00
asof_utc,2026-11-16 22:00:00+00:00,2026-11-17 22:00:00+00:00
cash_prompt,2026-11-18 00:00:00,2026-11-19 00:00:00
prompt,2027-02-16 00:00:00,2027-02-17 00:00:00
days,90,90
rate_date,2026-11-13 00:00:00,2026-11-16 00:00:00
rate,3.88,3.88
rate_utc,2026-11-16 14:00:00+00:00,2026-11-17 14:00:00+00:00


| Column | Meaning |
|---|---|
| `trade_date` (index) | observation date, an LME business day with both official prices |
| `cash`, `three_month`, `price_utc` | the two LME official prices, USD/t, and the instant they were readable |
| `asof_utc` | the anchor: New York close on the trade date |
| `cash_prompt`, `prompt`, `days` | the two LME prompts and the calendar days between them, over which the curve runs |
| `rate`, `eurusd` with `_date` and `_utc` | funding rate and euro, the date each describes and when it was published |
| `arrival`, `delivery_month` | the day the metal arrives and the month whose premium it is sold on |
| `premium_*`, `premium_du_*` | the premium contract of the arrival month: contract, trade date, settle, instant; duty-unpaid for Rotterdam |
| `us_tariff` | section 232 rate proclaimed by the trade date and in force on arrival |
| `cbam_published`, `cbam_price` | latest CBAM certificate price published by the trade date, EUR/tCO2e |
| `war_gulf`, `war_red_sea` | carrier war surcharges that apply to the route, USD per box |
| `bab_el_mandeb`, `traffic` | container-ship transits against 2019 at Bab el-Mandeb, and at the route's most disrupted passage |
| `staleness_days` | age of the oldest price leg on the row |

**What this dataset is not.** It is one merchant's copy of the LME's official prices, four premium futures that are often unchanged from one day to the next, a freight calibrated on one index and held flat, and policy files typed from the documents they cite. The timestamps are publication conventions. It is point-in-time for a daily decision at the New York close, and should not be used intraday.

## 9. Trade statistics

The check of notebook 03 uses monthly trade in HS 7601 declared by the importers. Each download keeps every release: a month Comtrade republishes comes back as a new row next to the old one.

In [39]:
sql.query("""
    SELECT file, COUNT(DISTINCT period) AS months, MIN(period) AS first_month, MAX(period) AS last_month,
           COUNT(DISTINCT partner) AS partners, COUNT(DISTINCT released) AS releases, CAST(MAX(ingested_at) AS DATE) AS ingested
    FROM raw_flows GROUP BY 1 ORDER BY 1
""")

,file,months,first_month,last_month,partners,releases,ingested
0,comext_eu_imports.csv,129,2016-01-01,2026-09-01,6,1,2026-11-18
1,comtrade_canada_exports.csv,32,2024-01-01,2026-08-01,50,32,2026-11-19
2,comtrade_japan_imports.csv,129,2016-01-01,2026-09-01,97,129,2026-11-19
3,comtrade_us_imports.csv,128,2016-01-01,2026-08-01,67,128,2026-11-19


In [40]:
# keys and impossible values: one row per file, partner, month and release; no negative tonnage
sql.query("""
    SELECT COUNT(*) AS rows, COUNT(*) - COUNT(DISTINCT (file, partner, period, released)) AS duplicates,
           COUNT(*) FILTER (WHERE tonnes IS NULL) AS missing, COUNT(*) FILTER (WHERE tonnes < 0) AS negative
    FROM raw_flows
""")

,rows,duplicates,missing,negative
0,10327,0,32,0


In [41]:
# Comtrade's release dates: how long after the end of the month each reporter first publishes it
releases = sql.query("SELECT reporter, period, MIN(first_released) AS first_released FROM flows WHERE first_released IS NOT NULL GROUP BY 1, 2")
releases["lag_days"] = (releases["first_released"] - (releases["period"] + pd.offsets.MonthEnd(0))).dt.days
releases.groupby("reporter")["lag_days"].describe()[["count", "min", "50%", "max"]]

,count,min,50%,max
reporter,,,,
canada_exports,32.00,35.00,51.50,136.00
japan,129.00,27.00,55.00,993.00
midwest,128.00,38.00,76.00,506.00


In [42]:
# Comext's convention against the download: the newest month on file must be out under the convention, the one after not
comext = sql.query("SELECT MAX(period) AS newest, CAST(MAX(ingested_at) AS DATE) AS downloaded FROM raw_flows WHERE file = 'comext_eu_imports.csv'").iloc[0]
out_at = comext["newest"] + pd.offsets.MonthEnd(0) + pd.Timedelta(days=c.COMEXT_LAG_DAYS)
next_at = comext["newest"] + pd.offsets.MonthEnd(2) + pd.Timedelta(days=c.COMEXT_LAG_DAYS)
assert out_at <= comext["downloaded"] < next_at
print(f"downloaded {comext['downloaded']:%d %b %Y}: {comext['newest']:%B %Y} readable from {out_at:%d %b} under the convention, the next month from {next_at:%d %b}")

downloaded 18 Nov 2026: September 2026 readable from 15 Nov under the convention, the next month from 16 Dec


- **Every source is on file to its newest month**: Comext and Japan to September 2026, the US to August; Canada's own exports from 2024, the window notebook 03 reads them for.
- **Comtrade gives each month its own release dates.** The US and Japan reach Comtrade a median 55 to 76 days after the month; the early months show lags of over a year because they were loaded into Comtrade's current database after the fact. Notebook 03 uses these dates; Comext's convention passes the test against its download.
- **32 rows have a value but no net weight**: totals for the world and small partners, none of them an origin of this project, and Canada's exports to the US in November 2025, which notebook 03 leaves out of the quarterly figures rather than counting as zero.

---
The dataset is ready: notebooks 02 and 03 read `routes`, `panel` and the trade tables and add no data operation of their own.